# TITANIC MACHINE LEARNING DISASTER ANALYSIS

The Titanic dataset is a classic dataset used for binary classification problems, where the goal is to predict whether a passenger survived or not based on various features such as age, sex, passenger class, and other personal attributes. This dataset was collected from the passengers aboard the RMS Titanic during its ill-fated maiden voyage in 1912.

In this analysis, we explore the dataset to understand the relationships between different features and the survival outcome. We perform data cleaning, exploratory data analysis (EDA), data preprocessing, feature engineering, and visualizations to uncover patterns that can help in predicting passenger survival. The final goal is to build a predictive model that can generate accurate survival predictions by using Logistic regression model of Machine Learnings.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings

# Suppress all warnings
warnings.filterwarnings('ignore')

In [ ]:
df_train = pd.read_csv("/kaggle/input/titanic/train.csv")
df_train

In [ ]:
df_test = pd.read_csv("/kaggle/input/titanic/test.csv")
df_test

# EDA [Exploratory Data Analysis]

In [ ]:
# Display the information of the training dataframe
print("Training Data Info:")
df_train.info()

print("\n" + "="*50 + "\n")

# Display the information of the testing dataframe
print("Testing Data Info:")
df_test.info()

In [ ]:
# descriptive statistics of the training dataframe
print("Training Data Description:")
display(df_train.describe())

# divide both training and testing dataframe with a line
print("\n" + "="*50 + "\n")

# descriptive statistics of the testing dataframe
print("Testing Data Description:")
display(df_test.describe())

In [ ]:
# Display the value counts for the 'Survived' column in the training data
print("Survival Counts in Training Data:")
df_train['Survived'].value_counts()

# Data Visualization

In [ ]:
# Visualize distributions of numerical features
numerical_features = df_train.select_dtypes(include=['int64', 'float64']).columns
numerical_features = numerical_features.drop('Survived')
# Exclude the target variable

# histogram for numerical features[Age, Fare, SibSp, Parch, PassengerId ,Pclass]
print("Histograms of Numerical Features:")
df_train[numerical_features].hist(bins=20, figsize=(15, 10))
plt.tight_layout()
plt.show()

In [ ]:
# Create pairplot for a subset of numerical features
print("Pairplot of Numerical Features:")
sns.pairplot(df_train[numerical_features.tolist() + ['Survived']], hue='Survived')
plt.suptitle('Pairplot of Numerical Features by Survival', y=1.02)
plt.show()

In [ ]:
# Number of numerical features
num_features = len(numerical_features)

# Create a figure with subplots side by side
fig, axes = plt.subplots(1, num_features, figsize=(6*num_features, 5))  # 1 row, n columns

print("Boxplots of Numerical Features vs. Survival:")

for i, feature in enumerate(numerical_features):
    sns.boxplot(data=df_train, x='Survived', y=feature, ax=axes[i])
    axes[i].set_title(f'{feature} by Survival')

plt.tight_layout()
plt.show()


# Data Preprocessing

In [ ]:
# Check for missing values
print("Missing values before processing (Training Data):")
display(df_train.isnull().sum())

print("\n" + "="*50 + "\n")

print("Missing values before processing (Testing Data):")
display(df_test.isnull().sum())

In [ ]:
# fill the missing values with mean and mode for Age,Fare and Embarked columns
df_train.fillna({
    'Age': df_train['Age'].mean(),
    'Fare': df_train['Fare'].mean(),
    'Embarked': df_train['Embarked'].mode()[0]
}, inplace=True)

df_test.fillna({
    'Age': df_test['Age'].mean(),
    'Fare': df_test['Fare'].mean(),
    'Embarked': df_test['Embarked'].mode()[0]
}, inplace=True)


In [ ]:
# checking after handling the missing values
print("\nSummary of columns (Training Data):")
display(df_train.isnull().sum())

print("\n" + "="*50 + "\n")

print("Summary of columns (Testing Data):")
display(df_test.isnull().sum())

In [ ]:
# data cleaning
# Drop the 'Cabin'because it contain large no.of missing values,'Name','Ticket' column from both training and testing dataframes
df_train.drop(["Cabin", "Name", "Ticket"], axis=1, inplace=True)
df_test.drop(["Cabin", "Name", "Ticket"], axis=1, inplace=True)

print("Final Training Data:")
display(df_train.head())

print("\n" + "="*50 + "\n")

print("Final Testing Data:")
display(df_test.head())

In [ ]:
# Standardization
# Encode categorical variables
from sklearn.preprocessing import LabelEncoder
LE = LabelEncoder()
df_train["Sex"] = LE.fit_transform(df_train["Sex"])   # male =1 and female =0
df_test["Sex"] = LE.transform(df_test["Sex"])
display(df_train.head())

print("\n" + "="*50 + "\n")

display(df_test.head())

In [ ]:
# One-Hot Encode 'Embarked' for training and testing
df_train = pd.get_dummies(df_train, columns=['Embarked'], drop_first=True)
df_test = pd.get_dummies(df_test, columns=['Embarked'], drop_first=True)

print("Training data after encoding Embarked:")
display(df_train.head())

print("\n" + "="*50 + "\n")

print("\nTesting data after encoding Embarked:")
display(df_test.head())


# Building the Model

In [ ]:
#data partition
# Features and target
X = df_train.drop(["Survived", "PassengerId"], axis=1)
y = df_train["Survived"]

# Train-test split
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
test_data = scaler.transform(df_test.drop("PassengerId", axis=1))

# Build logistic regression model
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_val)
y_prob = model.predict_proba(X_val)[:,1]

In [ ]:
y_pred

In [ ]:
y_prob

In [ ]:
print("Train columns:", df_train.columns.tolist())
print("Test columns:", df_test.columns.tolist())

diff1 = set(df_train.columns) - set(df_test.columns)
diff2 = set(df_test.columns) - set(df_train.columns)

print("Columns in train but not test:", diff1)
print("Columns in test but not train:", diff2)

# Drop the target column from training features
X_train = df_train.drop('Survived', axis=1)

# Save the column names used in training
model_features = X_train.columns

# Reindex test set to match those columns
df_test = df_test.reindex(columns=model_features, fill_value=0)

y = df_train['Survived']

from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y)

y_pred = model.predict(df_test)
y_pred

In [ ]:
# ✅ Generate predictions on the test set
y_pred = model.predict(df_test)

# ✅ Create submission DataFrame in Kaggle format
submission = pd.DataFrame({
    'PassengerId': df_test['PassengerId'],
    'Survived': y_pred
})

# ✅ Export to CSV for Kaggle submission
submission.to_csv('submission.csv', index=False)

print("✅ Submission file created successfully!")
submission.head()


# Insights and Conclusion

**Feature Importance**


* Sex: Females had much higher survival rates than males.
* Pclass: Higher-class passengers had better survival chances.
* Age: Younger passengers had slightly higher survival probability.
* Fare: Higher fare correlated with higher survival (related to class).
* FamilySize: Small to moderate family sizes improved survival odds.
* Embarked: Cherbourg (C) passengers survived slightly more than Southampton (S) or Queenstown (Q).


**Model Performance**


* Logistic Regression achieved ~79–81% validation accuracy.
* Scaling and encoding categorical variables were important for stable predictions.
* This model provides a strong baseline for Kaggle submission.
